In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from scipy.ndimage import distance_transform_edt
from matplotlib.animation import FuncAnimation, FFMpegWriter


# --- Load binned 2D data ---
data = np.load("prepared_data_2d/velocities_grid_binned_cropped.npz")
mag_grid = data["mag"]             # (nt, NY, NX)
mean_mag_grid = data["mean_mag"]
X = data["X"]
Y = data["Y"]
mask_grid = data["mask_grid"]
times = data["times"]

nt, NY, NX = mag_grid.shape
print(f"Loaded binned grid data: {nt} time steps, grid size {NX}x{NY}")


# --- Compute global color normalization ---
vals = mag_grid.ravel()
vals = vals[~np.isnan(vals)]

# --- Create figure for animation ---
fig, ax = plt.subplots(figsize=(10, 4))
initial = mag_grid[0]
initial[mask_grid] = np.nan
im = ax.imshow(initial, extent=[X.min(), X.max(), Y.min(), Y.max()],
               origin="lower", cmap="rainbow")
ax.set_title("t = 0")
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect(2)
cbar = fig.colorbar(im, ax=ax, orientation="vertical", fraction=0.03, pad=0.02)
cbar.set_label("|v|")

# --- Animation update function ---
def update(frame):
    vals = mag_grid[frame]
    vals[mask_grid] = np.nan
    im.set_data(vals)
    ax.set_title(f"timestep {frame}")
    return [im]

# --- Create animation ---
ani = FuncAnimation(fig, update, frames=nt, interval=100, blit=False)

# --- Save to MP4 (requires ffmpeg installed) ---
writer = FFMpegWriter(fps=10, bitrate=1800)
ani.save("velocity_animation.gif", writer=writer)
plt.close(fig)

Loaded binned grid data: 1600 time steps, grid size 500x207
